# Daily option-history scraper (IBKR)

Builds your own historical options dataset, one trading day at a time. For each symbol in `SYMBOLS` it records:

**Underlying:** open, high, low, close, and volume for the day (regular trading hours).

**Options:** the **10 strikes closest to the underlying's close**, calls and puts, at the **3 nearest expiries plus the furthest expiry**. For each contract:
- `iv_open`: IV at the open. IBKR's IV calculator backs it out of the option's 9:30 midpoint and the underlying's open price.
- `iv_close`: IBKR's model IV at the close. `iv_close_calc` is the same calculation as `iv_open`, applied to the closing prices, so the two can be compared directly.
- `open_interest`, `option_volume`, `delta`, `gamma`, `theta`, `vega`, plus the option's opening and closing midpoints and its bid/ask.

Each run appends to `~/OptionHistory/data/options_daily.csv` and `underlying_daily.csv`. Your Trading folder has a shortcut to that folder, `option_history_data`. Running it again on the same day replaces that day's rows for the symbol instead of adding to them.

**Before each run:**
1. TWS or IB Gateway is running and logged in. In *Global Configuration → API → Settings*: enable *ActiveX and Socket Clients*, and add `127.0.0.1` to the trusted IPs.
2. Set `PORT` to your setup: TWS live is 7496, TWS paper 7497, Gateway live 4001, Gateway paper 4002.
3. You need market-data subscriptions for US equities and OPRA options. Without them, greeks, IV, and OI come back empty.

The scheduled run happens Monday–Friday at 4:30 pm ET (launchd job `com.nicksanso.option-scraper`, script `~/OptionHistory/run_option_scraper.sh`). The real files live in `~/OptionHistory` because macOS doesn't let background jobs read the Desktop. The copy in your Trading folder is a shortcut to them, so edits made there still apply.

In [ ]:
import math, os, shutil, time, logging
from collections import Counter
from datetime import datetime, date
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
from ib_async import IB, Stock, Index, Option, util
from ib_async.client import Client

# ---------------- settings ----------------
SYMBOLS = ["SPY"]                       # add more tickers here, e.g. ["SPY", "QQQ", "AAPL", "SPX"]
HOST, PORT, CLIENT_ID = "127.0.0.1", 7496, 41
MARKET_DATA_TYPE = 2                    # 2 = frozen: live while open, last values at the close after hours (4 = delayed-frozen if no subscription)

N_STRIKES = 10                          # strikes closest to the underlying close
N_NEAR_EXPIRIES = 3
N_FAR_EXPIRIES = 1
RIGHTS = ["C", "P"]

INDEX_EXCHANGES = {"SPX": "CBOE", "XSP": "CBOE", "VIX": "CBOE", "NDX": "NASDAQ", "RUT": "RUSSELL"}
PREFERRED_TRADING_CLASS = {"SPX": "SPXW", "NDX": "NDXP", "RUT": "RUTW"}   # pick the PM-settled weeklies

TICK_WAIT_SECONDS = 30                  # max wait for greeks/OI to arrive per batch
MKT_DATA_BATCH = 40                     # stays under IBKR's default 100 simultaneous market-data lines

DATA_DIR = Path.home() / "OptionHistory" / "data"
OPTIONS_CSV = DATA_DIR / "options_daily.csv"
UNDERLYING_CSV = DATA_DIR / "underlying_daily.csv"
ET = ZoneInfo("America/New_York")

DATA_DIR.mkdir(parents=True, exist_ok=True)
log = logging.getLogger("option_scraper")
log.setLevel(logging.INFO)
if not log.handlers:
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    for h in (logging.FileHandler(DATA_DIR / "scrape.log"), logging.StreamHandler()):
        h.setFormatter(fmt)
        log.addHandler(h)

## Connect to TWS / IB Gateway

In [ ]:
# ib_async sends the IV-calculator options as a count plus a list, which current TWS rejects
# with error 320 ("Please use 'Key=Value' format"). Send them as a single "k=v;" string instead.
def _calc_iv(self, reqId, contract, optionPrice, underPrice, implVolOptions):
    self.send(54, 3, reqId, contract, optionPrice, underPrice,
              "".join(f"{t.tag}={t.value};" for t in implVolOptions))
Client.calculateImpliedVolatility = _calc_iv

util.startLoop()
ib = IB()
ib.RequestTimeout = 30                  # so no single request can hang the daily run
ib.connect(HOST, PORT, clientId=CLIENT_ID, timeout=30, readonly=True)
ib.reqMarketDataType(MARKET_DATA_TYPE)
log.info("Connected to IBKR on port %s", PORT)

## Helpers

In [ ]:
def num(x):
    """Normalize IBKR's None/nan 'no data' values to NaN."""
    try:
        x = float(x)
    except (TypeError, ValueError):
        return math.nan
    return x if math.isfinite(x) else math.nan


def price(x):
    """Quotes use -1 / 0 for 'no quote'."""
    x = num(x)
    return x if x > 0 else math.nan


def bar_date(b):
    return b.date.date() if isinstance(b.date, datetime) else b.date


def daily_bars(contract, what):
    return ib.reqHistoricalData(contract, endDateTime="", durationStr="5 D", barSizeSetting="1 day",
                                whatToShow=what, useRTH=True, formatDate=1)


def underlying_contract(sym):
    c = Index(sym, INDEX_EXCHANGES[sym], "USD") if sym in INDEX_EXCHANGES else Stock(sym, "SMART", "USD")
    ib.qualifyContracts(c)
    if not c.conId:
        raise ValueError(f"Could not qualify {sym}")
    return c


def underlying_day(und):
    bars = daily_bars(und, "TRADES")
    if not bars:
        raise RuntimeError(f"No daily bars for {und.symbol}")
    b = bars[-1]
    d = bar_date(b)
    now = datetime.now(ET)
    if d == now.date() and (now.hour, now.minute) < (16, 0):
        log.warning("%s: market still open, so today's 'close' is provisional", und.symbol)
    return {"date": d.isoformat(), "symbol": und.symbol,
            "und_open": b.open, "und_high": b.high, "und_low": b.low,
            "und_close": b.close, "und_volume": b.volume}


def pick_expiries(und, record_date):
    chains = ib.reqSecDefOptParams(und.symbol, "", und.secType, und.conId)
    exps = sorted({e for c in chains if c.exchange == "SMART" for e in c.expirations})
    exps = [e for e in exps if e > record_date.replace("-", "")]   # drop anything expiring today or earlier
    near = exps[:N_NEAR_EXPIRIES]
    far = [e for e in exps[-N_FAR_EXPIRIES:] if e not in near] if N_FAR_EXPIRIES else []
    return near + far


def option_contracts(sym, expiry, und_close):
    details = ib.reqContractDetails(Option(sym, expiry, exchange="SMART", currency="USD"))
    cons = [d.contract for d in details if d.contract.right in RIGHTS]
    cons = [c for c in cons if c.multiplier in ("100", "")] or cons      # skip adjusted (non-standard) contracts
    if not cons:
        return []
    classes = Counter(c.tradingClass for c in cons)
    pref = PREFERRED_TRADING_CLASS.get(sym)
    tc = pref if pref in classes else (sym if sym in classes else classes.most_common(1)[0][0])
    cons = [c for c in cons if c.tradingClass == tc]
    strikes = set(sorted({c.strike for c in cons}, key=lambda k: abs(k - und_close))[:N_STRIKES])
    return sorted((c for c in cons if c.strike in strikes), key=lambda c: (c.right, c.strike))


def open_interest(t):
    oi = num(t.callOpenInterest if t.contract.right == "C" else t.putOpenInterest)
    return oi if not math.isnan(oi) else num(t.openInterest)


def market_data(contracts):
    """Greeks, IV, OI, and volume at the close, keyed by conId."""
    out = {}
    for i in range(0, len(contracts), MKT_DATA_BATCH):
        batch = contracts[i:i + MKT_DATA_BATCH]
        tickers = [ib.reqMktData(c, "101", False, False) for c in batch]   # 101 = open interest
        deadline = time.time() + TICK_WAIT_SECONDS
        while time.time() < deadline:
            ib.sleep(0.5)
            if all(t.modelGreeks is not None and not math.isnan(open_interest(t)) for t in tickers):
                break
        for c, t in zip(batch, tickers):
            g = t.modelGreeks
            out[c.conId] = {
                "iv_close": num(g.impliedVol) if g else math.nan,
                "delta": num(g.delta) if g else math.nan,
                "gamma": num(g.gamma) if g else math.nan,
                "theta": num(g.theta) if g else math.nan,
                "vega": num(g.vega) if g else math.nan,
                "model_und_price": num(g.undPrice) if g else math.nan,
                "open_interest": open_interest(t),
                "option_volume": num(t.volume),
                "bid": price(t.bid),
                "ask": price(t.ask),
            }
            ib.cancelMktData(c)
    return out


def option_open_close(c, record_date):
    """Option midpoint at the 9:30 open and the 16:00 close on record_date.
    IBKR has no daily bars for options. 4-hour bars split the day into 9:30-12:00, 12:00-16:00 and
    16:00-16:15 (options trade until 16:15). Dropping the 16:00 bar leaves the first bar's open (9:30)
    and the 12:00 bar's close (16:00), which lines up with the stock's 4pm close."""
    bars = ib.reqHistoricalData(c, endDateTime="", durationStr="2 D", barSizeSetting="4 hours",
                                whatToShow="MIDPOINT", useRTH=True, formatDate=1)
    day = [b for b in bars
           if b.date.astimezone(ET).date().isoformat() == record_date and b.date.astimezone(ET).hour < 16]
    if not day:
        return math.nan, math.nan
    return price(day[0].open), price(day[-1].close)


def implied_vol(c, opt_price, und_price):
    if not (opt_price > 0 and und_price > 0):
        return math.nan
    try:
        comp = ib.calculateImpliedVolatility(c, opt_price, und_price)
        return num(comp.impliedVol) if comp else math.nan
    except Exception as e:
        log.debug("IV calc failed for %s: %s", c.localSymbol, e)
        return math.nan

## Scrape

In [ ]:
OPTION_COLUMNS = [
    "date", "symbol", "und_open", "und_high", "und_low", "und_close", "und_volume",
    "expiry", "dte", "strike", "right",
    "iv_open", "iv_close", "iv_close_calc", "open_interest", "option_volume",
    "delta", "gamma", "theta", "vega",
    "opt_open_mid", "opt_close_mid", "bid", "ask", "model_und_price",
    "con_id", "local_symbol", "trading_class", "scraped_at",
]


def scrape_symbol(sym):
    und = underlying_contract(sym)
    u = underlying_day(und)
    rec = u["date"]
    expiries = pick_expiries(und, rec)
    contracts = [c for e in expiries for c in option_contracts(sym, e, u["und_close"])]
    log.info("%s %s: close %.2f, expiries %s, %d contracts", sym, rec, u["und_close"], expiries, len(contracts))

    md_by_con = market_data(contracts)
    rec_date = date.fromisoformat(rec)
    rows = []
    for c in contracts:
        o_mid, c_mid = option_open_close(c, rec)
        exp = c.lastTradeDateOrContractMonth
        rows.append({
            **u,
            "expiry": exp,
            "dte": (datetime.strptime(exp, "%Y%m%d").date() - rec_date).days,
            "strike": c.strike, "right": c.right,
            "iv_open": implied_vol(c, o_mid, u["und_open"]),
            "iv_close_calc": implied_vol(c, c_mid, u["und_close"]),
            "opt_open_mid": o_mid, "opt_close_mid": c_mid,
            **md_by_con.get(c.conId, {}),
            "con_id": c.conId, "local_symbol": c.localSymbol, "trading_class": c.tradingClass,
            "scraped_at": datetime.now(ET).isoformat(timespec="seconds"),
        })
    return u, rows


underlying_rows, option_rows, failed = [], [], []
try:
    for sym in SYMBOLS:
        try:
            u, rows = scrape_symbol(sym)
            underlying_rows.append(u)
            option_rows.extend(rows)
        except Exception:
            log.exception("Scrape failed for %s", sym)
            failed.append(sym)
finally:
    ib.disconnect()

options_today = pd.DataFrame(option_rows).reindex(columns=OPTION_COLUMNS)
underlying_today = pd.DataFrame(underlying_rows)
options_today.head(20)

## Append to the history files

In [ ]:
def append_replace_day(new, path, keys):
    """Append new rows to the CSV. A re-run replaces all earlier rows for the same date and symbol,
    so strikes from an earlier run that day don't linger after the price has moved."""
    if new.empty:
        return 0
    combined = new
    if path.exists():
        shutil.copy2(path, path.with_suffix(".csv.bak"))       # one-step undo in case anything goes wrong
        old = pd.read_csv(path, dtype={"expiry": str, "date": str})
        rerun = old.set_index(["date", "symbol"]).index.isin(new.set_index(["date", "symbol"]).index)
        combined = pd.concat([old[~rerun], new], ignore_index=True)
    combined = combined.sort_values(keys)
    tmp = path.with_suffix(".tmp")
    combined.to_csv(tmp, index=False)
    os.replace(tmp, path)                                     # atomic, so a crash can't leave a half-written file
    return len(combined)


n_opt = append_replace_day(options_today, OPTIONS_CSV, ["date", "symbol", "expiry", "strike", "right"])
n_und = append_replace_day(underlying_today, UNDERLYING_CSV, ["date", "symbol"])
log.info("Saved %d option rows today (%d total), %d underlying rows total",
         len(options_today), n_opt, n_und)

missing = options_today[["iv_close", "delta", "open_interest"]].isna().mean() if len(options_today) else None
if missing is not None and (missing > 0.5).any():
    log.warning("More than half the rows are missing some fields, so check your market-data subscriptions:\n%s", missing)
if failed:
    raise RuntimeError(f"Failed symbols: {failed}")      # non-zero exit so the scheduled run shows up as failed in launchd.log